# 03 · Build a neural network

## What you will build

A multilayer network made of your own dense layers: the layer computation, the weight initialisation, and a parameter counter. You will wire a tiny network by hand that solves XOR, then build a random network for the spiral data and see that, untrained, it is no better than guessing.

## Learning objectives

- Write a dense layer as one matrix product: $Z = XW + b$.
- Initialise weights with He initialisation, and count a network's parameters.
- Explain why a hidden layer with a nonlinear activation can solve XOR.
- Recognise that random weights give random predictions, which is why we train.

## Dependencies

Tier 1: NumPy and Matplotlib only.

### How to work through this notebook

Run the cells from the top; type each **✍️ Your turn** function and run its `practice` check. Once your function matches the reference, the rest of the notebook uses it.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.

def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class: X (n, 2), integer labels y (n,)."""
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 02: activations.

def relu(Z):
    return np.maximum(0.0, Z)


def sigmoid(Z):
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax; used here only to turn the output layer into probabilities."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)

## 1 · A layer is many neurons at once

A layer with $m$ neurons, each reading the same $n$ inputs, has a weight matrix $W$ of shape `(n, m)` (one column per neuron) and a bias vector $b$ of shape `(m,)`. For a batch $X$ of shape `(batch, n)`:

$$Z = X W + b \quad \text{has shape} \quad (\text{batch}, m).$$

The bias is added to every row by broadcasting.

### ✍️ Your turn: `dense_forward`

Return `Z = X @ W + b`. `X`: `(batch, n_in)`, `W`: `(n_in, n_out)`, `b`: `(n_out,)`; the result has shape `(batch, n_out)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def dense_forward_reference(X, W, b):
    return X @ W + b

In [ ]:
def dense_forward(X, W, b):
    """Z = X @ W + b."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
X_small = np.array([[1.0, 2.0], [0.0, -1.0], [3.0, 0.5]])
W_small = np.array([[0.5, -1.0, 0.0], [2.0, 0.25, -0.5]])
b_small = np.array([0.1, 0.0, -0.2])
dense_forward = practice("dense_forward", dense_forward, dense_forward_reference, cases=[
    (X_small, W_small, b_small),
    (np.ones((1, 2)), W_small, np.zeros(3)),
])

## 2 · Initialising the weights

If every weight starts at the same value, every neuron in a layer computes the same thing and receives the same update, forever. Random weights break that symmetry. Their scale matters too: too large and signals explode as they pass through layers; too small and they fade away. **He initialisation**, designed for ReLU layers, draws each weight from a normal distribution with standard deviation $\sqrt{2 / n_\text{in}}$, and starts biases at zero.

A network is a list of layers, each a dictionary `{"W": ..., "b": ...}`. For `sizes = [2, 16, 3]` there are two layers: `2 → 16` and `16 → 3`.

### ✍️ Your turn: `init_network`

Return a list of layers for consecutive pairs in `sizes`: `{"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}`. Use the `rng` you are given, in layer order, so results are reproducible.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def init_network_reference(sizes, rng):
    return [{"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
            for n_in, n_out in zip(sizes[:-1], sizes[1:])]

In [ ]:
def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 16, 3]."""
    # ✍️ Your code here: loop over zip(sizes[:-1], sizes[1:])
    raise NotImplementedError

In [ ]:
# Both versions get a fresh generator with the same seed, so they draw identical numbers.
def init_network_checked(sizes, seed):
    return init_network(sizes, np.random.default_rng(seed))

def init_network_reference_checked(sizes, seed):
    return init_network_reference(sizes, np.random.default_rng(seed))

_checked = practice("init_network", init_network_checked, init_network_reference_checked, cases=[
    ([2, 4, 3], 0), ([3, 5, 5, 2], 1),
])
if _checked is init_network_reference_checked:
    init_network = init_network_reference

### Counting parameters

Each layer has $n_\text{in} \times n_\text{out}$ weights plus $n_\text{out}$ biases. A `[2, 16, 3]` network therefore has $2 \cdot 16 + 16 + 16 \cdot 3 + 3 = 99$ parameters.

### ✍️ Your turn: `count_parameters`

Return the total number of weights and biases in `network` (an `int`). Hint: `layer["W"].size`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def count_parameters_reference(network):
    return int(sum(layer["W"].size + layer["b"].size for layer in network))

In [ ]:
def count_parameters(network):
    """Total number of trainable numbers in the network."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
count_parameters = practice("count_parameters", count_parameters, count_parameters_reference, cases=[
    (init_network_reference([2, 16, 3], np.random.default_rng(0)),),
    (init_network_reference([4, 8, 8, 2], np.random.default_rng(0)),),
])
print("parameters in [2, 16, 3]:", count_parameters(init_network([2, 16, 3], np.random.default_rng(0))))

## 3 · Two hidden neurons solve XOR

A hidden layer lets the network bend its boundary. With ReLU hidden units $h_1 = \max(0, x_1 + x_2)$ and $h_2 = \max(0, x_1 + x_2 - 1)$, the output $h_1 - 2h_2$ is exactly XOR: it is 0 for $(0,0)$, 1 for $(0,1)$ and $(1,0)$, and $2 - 2 = 0$ for $(1,1)$. Below, those weights are written as a network and run through **your** `dense_forward`.

In [ ]:
xor_net = [
    {"W": np.array([[1.0, 1.0], [1.0, 1.0]]), "b": np.array([0.0, -1.0])},
    {"W": np.array([[1.0], [-2.0]]), "b": np.array([0.0])},
]
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
hidden = relu(dense_forward(X_xor, xor_net[0]["W"], xor_net[0]["b"]))
output = dense_forward(hidden, xor_net[1]["W"], xor_net[1]["b"]).ravel()
print("hidden activations:\n", hidden)
print("XOR output:", output)

### 🧪 Try it

Delete the ReLU (use `hidden = dense_forward(...)` without `relu`) and re-run. Why does the network lose XOR? Then put it back.

## 4 · A random network on the spirals

Now a real architecture for the three-class spiral data from notebook 01: `[2, 64, 3]`. The forward pass is a ReLU hidden layer and a softmax output (softmax turns the 3 output numbers into probabilities; notebook 04 builds it properly). With random weights, how accurate is it?

In [ ]:
X, y = make_spirals(100, 3, 0.2, np.random.default_rng(1))
network = init_network([2, 64, 3], np.random.default_rng(2))

def run(network, X):
    A = relu(dense_forward(X, network[0]["W"], network[0]["b"]))
    return softmax(dense_forward(A, network[1]["W"], network[1]["b"]))

random_accuracy = float(np.mean(run(network, X).argmax(axis=1) == y))
print(f"untrained accuracy: {random_accuracy:.2f} (chance is about 0.33)")

In [ ]:
grid = np.linspace(-1.1, 1.1, 150)
gx, gy = np.meshgrid(grid, grid)
regions = run(network, np.column_stack([gx.ravel(), gy.ravel()])).argmax(axis=1).reshape(gx.shape)
plt.figure(figsize=(4.5, 4.5))
plt.contourf(gx, gy, regions, levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis", alpha=0.35)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=12, edgecolors="k", linewidths=0.3)
plt.title("random weights: the regions ignore the spirals")
plt.show()

### 🧪 Try it

Change the seed in `np.random.default_rng(2)` or the hidden width `64` and re-run the last two cells. The regions change, but the accuracy stays near chance: structure has to be **learned**.

## 5 · Put it together

In [ ]:
check_net = init_network([2, 64, 3], np.random.default_rng(3))
assert [layer["W"].shape for layer in check_net] == [(2, 64), (64, 3)]
assert all(np.all(layer["b"] == 0) for layer in check_net)
assert count_parameters(check_net) == 2 * 64 + 64 + 64 * 3 + 3
assert np.allclose(output, [0.0, 1.0, 1.0, 0.0]), "the hand-built network should compute XOR"
assert random_accuracy < 0.6, "an untrained network should be close to chance"
print("✅ Layers, initialisation and parameter counts work; XOR is solved by hand; training is next.")

## Exercises

1. **Sigmoid hidden units.** Rebuild `run` with `sigmoid` instead of `relu` in the hidden layer. Does the untrained accuracy change much? Why might sigmoid layers train more slowly in deep networks?
2. **Deeper.** Initialise `[2, 32, 32, 3]` and count its parameters by hand before running `count_parameters`.
3. **Zero initialisation.** Set every weight to zero and run the network. What do all the hidden units output, and why would training never separate them?
4. **Open question.** The XOR network was solved by hand. On the spirals there are 291 numbers to set. How would you find them? (Notebooks 05–08.)

## Common errors and debugging

- **Shapes**: `W` is `(n_in, n_out)`. If you wrote `(n_out, n_in)`, `X @ W` fails or silently computes the wrong thing for square layers.
- **All hidden units identical**: you initialised every weight to the same value.
- **Exploding outputs**: a weight scale of 1.0 in a wide layer; use $\sqrt{2/n_\text{in}}$.

## Summary

A layer is a matrix product plus a bias; a network is a list of layers with nonlinear activations between them. He initialisation keeps signals at a sensible scale. Hidden layers let a network draw bent boundaries, which is how two ReLU units solve XOR, but with random weights a network is just guessing.

## Next

**04 · Forward propagation**: build the full forward pass with a softmax output and a cache for training.